In [ ]:
# ============================================================
# STEP 1: Overview & Setup
# Description: 
# This script iterates through all building folders listed in 
# your EPC dataset and replaces the file:
#     copied_cons.constrdb
# inside specific model folders (windows, walls, roof, fab)
# with the corresponding upgraded construction database.
#
# Logic:
# - Read EPC CSV
# - Loop through BUILDING_REFERENCE_NUMBER
# - Check if relevant model folders exist
# - If found, replace the constrdb file with the correct upgrade
# - Skip buildings/folders that do not exist
#
# Notes:
# - Safe overwrite (no delete step needed)
# - Prints progress for debugging
# ============================================================

import os
import shutil
import pandas as pd

# File paths
epc_file = "/Users/jakegehrung/Desktop/data_raw/data_processed/fintry_epc_NPV_IRR_update.csv"
baseline_root = "/Users/jakegehrung/Desktop/data_raw/baseline_models"

# Replacement files
replacement_files = {
    "windows": "/Users/jakegehrung/Desktop/data_raw/cons/contruction_database_windows_upgraded/copied_cons.constrdb",
    "walls": "/Users/jakegehrung/Desktop/data_raw/cons/contruction_database_walls_upgraded/copied_cons.constrdb",
    "roof": "/Users/jakegehrung/Desktop/data_raw/cons/contruction_database_roof_upgraded/copied_cons.constrdb",
    "fab": "/Users/jakegehrung/Desktop/data_raw/cons/contruction_database_all_upgraded/copied_cons.constrdb"
}

# Model folder name patterns
model_types = {
    "windows": ["SemiDetached_2F_windows", "Detached_2F_windows"],
    "walls": ["SemiDetached_2F_walls", "Detached_2F_walls"],
    "roof": ["SemiDetached_2F_roof", "Detached_2F_roof"],
    "fab": ["SemiDetached_2F_fab", "Detached_2F_fab"]
}

In [ ]:
# ============================================================
# STEP 2: Load EPC Dataset
# Description:
# Loads the EPC CSV and extracts BUILDING_REFERENCE_NUMBER list
# ============================================================

df = pd.read_csv(epc_file)

# Ensure IDs are strings (important for folder matching)
building_ids = df["BUILDING_REFERENCE_NUMBER"].astype(str).tolist()

print(f"Loaded {len(building_ids)} building IDs")

Loaded 117 building IDs


In [ ]:
# ============================================================
# STEP 3: Define File Replacement Function
# Description:
# Replaces copied_cons.constrdb inside the dbs folder
# ============================================================

def replace_constrdb(target_folder, replacement_file):
    dbs_path = os.path.join(target_folder, "dbs")
    target_file = os.path.join(dbs_path, "copied_cons.constrdb")

    if not os.path.exists(dbs_path):
        print(f"  ⚠️ No 'dbs' folder: {target_folder}")
        return

    if not os.path.exists(target_file):
        print(f"  ⚠️ Missing target file: {target_file}")
        return

    try:
        shutil.copy(replacement_file, target_file)
        print(f"  ✅ Replaced: {target_file}")
    except Exception as e:
        print(f"  ❌ Error replacing file in {target_folder}: {e}")

In [ ]:
# ============================================================
# STEP 4: Process All Buildings
# Description:
# Loops through each building and applies replacements
# based on detected model folders
# ============================================================

for building_id in building_ids:
    building_path = os.path.join(baseline_root, building_id)

    if not os.path.exists(building_path):
        print(f"\nSkipping missing building folder: {building_id}")
        continue

    print(f"\nProcessing building: {building_id}")

    # Loop through each model type (windows, walls, etc.)
    for model_key, folder_names in model_types.items():
        found = False

        for folder_name in folder_names:
            model_path = os.path.join(building_path, folder_name)

            if os.path.exists(model_path):
                print(f" Found {model_key} folder: {folder_name}")
                replace_constrdb(model_path, replacement_files[model_key])
                found = True

        if not found:
            print(f"  ⏭️ No {model_key} folder found")


Processing building: 1001829067
 Found windows folder: SemiDetached_2F_windows
  ✅ Replaced: /Users/jakegehrung/Desktop/data_raw/baseline_models/1001829067/SemiDetached_2F_windows/dbs/copied_cons.constrdb
 Found walls folder: SemiDetached_2F_walls
  ✅ Replaced: /Users/jakegehrung/Desktop/data_raw/baseline_models/1001829067/SemiDetached_2F_walls/dbs/copied_cons.constrdb
 Found roof folder: SemiDetached_2F_roof
  ✅ Replaced: /Users/jakegehrung/Desktop/data_raw/baseline_models/1001829067/SemiDetached_2F_roof/dbs/copied_cons.constrdb
 Found fab folder: SemiDetached_2F_fab
  ✅ Replaced: /Users/jakegehrung/Desktop/data_raw/baseline_models/1001829067/SemiDetached_2F_fab/dbs/copied_cons.constrdb

Processing building: 1001951858
 Found windows folder: Detached_2F_windows
  ✅ Replaced: /Users/jakegehrung/Desktop/data_raw/baseline_models/1001951858/Detached_2F_windows/dbs/copied_cons.constrdb
 Found walls folder: Detached_2F_walls
  ✅ Replaced: /Users/jakegehrung/Desktop/data_raw/baseline_models

In [ ]:
# ============================================================
# STEP 5: Summary (Optional)
# Description:
# Simple completion message for clarity
# ============================================================

print("\n🎉 Processing complete for all buildings.")


🎉 Processing complete for all buildings.
